# Emerging Technologies
### Nathan Carr - G00410214

## Introduction

This notebook is my submission for the Emerging Technologies module at ATU Galway (2025/26).

The main topic is the Deutsch–Jozsa algorithm, a quantum algorithm that can determine whether a Boolean function is constant or balanced using only one query, something a classical algorithm cannot always do in one step.

Problems 1 and 2 deal with the classical side: generating random constant/balanced functions and writing a classical algorithm to identify them. Problems 3, 4, and 5 move into quantum computing, building up from simple 1-bit oracles in Qiskit all the way to the full Deutsch–Jozsa circuit for 4-bit functions.

Each problem follows the same structure: background explaining the theory, implementation notes, a demonstration with output, a discussion of what the results mean, and references.

## Problem 1: Generating Random Boolean Functions

### Background

The Deutsch–Jozsa algorithm works with Boolean functions that take a fixed number of inputs and return a single Boolean output.

For this problem, the function takes **four Boolean inputs**, meaning there are:

\[
2^4 = 16
\]

possible input combinations.

The function is guaranteed to be either:

- **constant**: returns the same value (always `True` or always `False`)
- **balanced**: returns `True` for exactly half of the inputs (8 out of 16) and `False` for the rest

The goal is to generate a random function that satisfies this promise.

### Implementation

To solve this, I first generate all 16 possible input combinations using `itertools.product`.

Then:

- If the function is **constant**, I assign the same value (`True` or `False`) to all inputs.
- If the function is **balanced**, I randomly select 8 input combinations and assign them `True`, with the remaining inputs assigned `False`.

The function is stored as a lookup table (dictionary), and a Python function is returned that retrieves the output for a given input.

In [1]:
import random
from itertools import product

def random_constant_balanced(seed: int | None = None):
    """
    Return a random 4-input Boolean function that is either constant or balanced.

    Constant: always False or always True.
    Balanced: True for exactly 8 of the 16 possible inputs.
    """
    rng = random.Random(seed)
    inputs = list(product([False, True], repeat=4))  # 16 input tuples

    if rng.choice([True, False]):  # balanced
        true_inputs = set(rng.sample(inputs, k=8))   # choose exactly half
        table = {x: (x in true_inputs) for x in inputs}
    else:  # constant
        const_value = rng.choice([False, True])
        table = {x: const_value for x in inputs}

    def f(a: bool, b: bool, c: bool, d: bool) -> bool:
        return table[(a, b, c, d)]

    # Helpful for demonstration/testing in your notebook
    f.truth_table = table  # type: ignore[attr-defined]

    return f


### Demonstration

To verify that the function satisfies the required conditions, I count how many times it returns `True`.

For a valid function, the number of `True` outputs should always be:

- 0 (constant False)
- 16 (constant True)
- 8 (balanced)

In [2]:
# Generate a few functions and verify they match the promise
for s in range(5):
    f = random_constant_balanced(seed=s)
    num_true = sum(f.truth_table.values()) # type: ignore
    print(s, num_true)  # must be 0, 8, or 16


0 16
1 8
2 8
3 8
4 8


### Discussion

The output confirms that every generated function is either 0, 8, or 16 `True` values, never anything in between, which is exactly what the Deutsch–Jozsa promise requires.

One thing I noticed is that the generator picks constant vs balanced with a 50/50 chance before randomising within that category. That means you are just as likely to get a constant function as a balanced one, which feels fair for testing purposes.

In a real quantum computing scenario the oracle would be a black box, you would not know in advance whether it is constant or balanced, which is the whole point of needing an algorithm to figure it out. The seeded version here is mainly useful because it makes results reproducible when testing.

### References

- Deutsch, D. (1985). *Quantum Theory, the Church–Turing Principle and the Universal Quantum Computer*. Proceedings of the Royal Society of London A, 400, 97–117.  
  The paper that introduced the idea of a quantum oracle and the constant/balanced problem. Worth reading even if the maths is dense, the first few pages give good intuition.

- Nielsen, M. A., & Chuang, I. L. (2010). *Quantum Computation and Quantum Information*. Cambridge University Press.  
  The standard textbook for this area. Chapter 1 covers the Deutsch–Jozsa problem clearly and explains why it matters as a demonstration of quantum advantage.

- Python Documentation, *itertools.product*:  
  https://docs.python.org/3/library/itertools.html#itertools.product  
  Used to generate all 16 input combinations for a 4-bit Boolean function without writing nested loops.

## Problem 2: Classical Testing for Function Type

### Background

In this problem, the goal is to determine whether a given Boolean function is **constant** or **balanced**.

The function is guaranteed to follow the same rules as in Problem 1:

- constant = always returns the same value  
- balanced = returns `True` for exactly half of the inputs  

Classically, we do this by calling the function with different inputs and observing the outputs.

### Implementation

To solve this, I evaluate the function on different input combinations.

- I store the result of the first function call.
- Then I keep checking new inputs:
  - If I ever see a different output, the function must be **balanced**.
  - If I continue seeing the same output, I keep checking until I can be certain it is **constant**.

Since there are 16 possible inputs, I loop through them systematically using `itertools.product`.

In [3]:
from itertools import product

def determine_constant_balanced(f) -> str:
    """
    Determine whether a promised constant/balanced 4-input Boolean function is
    "constant" or "balanced".

    Worst-case calls to f: 9 (guarantees 100% certainty under the promise).
    """
    first = None

    for i, x in enumerate(product([False, True], repeat=4), start=1):
        y = f(*x)

        if first is None:
            first = y
        elif y != first:
            return "balanced"

        # After 9 identical outputs, the function cannot be balanced (only 8 of each)
        if i == 9:
            return "constant"

    # With the promise, execution should always return before this.
    raise RuntimeError("Promise violated: function is neither constant nor balanced.")


### Efficiency

To be **100% certain**, a classical algorithm may need to evaluate the function multiple times.

In the worst case:

- A balanced function has 8 `True` and 8 `False` outputs.
- It is possible to observe the same output for the first 8 inputs and still not know if the function is constant or balanced.

Because of this, we must check **one more input** to be certain.

So the maximum number of function calls required is:

\[
2^{n-1} + 1
\]

For \(n = 4\):

\[
2^3 + 1 = 9
\]

This means a classical solution may require up to **9 evaluations** to guarantee the correct answer.

### Demonstration

To test the solution, I generate functions using the method from Problem 1 and apply the classification function.

The output shows:

- the number of `True` values (0, 8, or 16)
- the predicted result (`constant` or `balanced`)

This confirms that the function correctly identifies the type in all cases.

In [4]:
# Quick demo using Problem 1 generator
for s in range(6):
    f = random_constant_balanced(seed=s)
    print(f"seed={s:2d}  trues={sum(f.truth_table.values()):2d}  classified={determine_constant_balanced(f)}") # type: ignore

seed= 0  trues=16  classified=constant
seed= 1  trues= 8  classified=balanced
seed= 2  trues= 8  classified=balanced
seed= 3  trues= 8  classified=balanced
seed= 4  trues= 8  classified=balanced
seed= 5  trues=16  classified=constant


### Discussion

The classifier works correctly in all cases and the early exit helps a lot in practice, if the function returns different values on the first two inputs you check, you can stop immediately and return `"balanced"`.

The worst case is still the interesting part though. If a balanced function happens to return the same value for its first 8 inputs (which is perfectly allowed), there is no way to tell it apart from a constant function until you check one more input. That ninth call is what gives you certainty. This is not a flaw in the implementation, it is a fundamental ceiling for any classical approach to this problem.

This is what makes the Deutsch–Jozsa algorithm worth studying. The quantum version does not query individual inputs one at a time, it evaluates all inputs at once using superposition and gets the answer in a single oracle call. Problem 5 demonstrates this directly.

### References

- Deutsch, D. (1985). *Quantum Theory, the Church–Turing Principle and the Universal Quantum Computer*. Proceedings of the Royal Society of London A, 400, 97–117.  
  Useful context for why the classical problem matters, this paper is the reason we care about the constant/balanced distinction in the first place.

- Nielsen, M. A., & Chuang, I. L. (2010). *Quantum Computation and Quantum Information*. Cambridge University Press.  
  Chapter 1 explains the classical complexity of the Deutsch–Jozsa problem and why 2^(n−1) + 1 is the worst-case bound for a deterministic classical algorithm.

- IBM Quantum, *Deutsch's Algorithm*:  
  https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm  
  Helpful for understanding the classical problem that the quantum algorithm is designed to solve faster.

- Python Documentation, *itertools.product*:  
  https://docs.python.org/3/library/itertools.html#itertools.product  
  Used to loop through all 16 input combinations systematically.

## Problem 3: Quantum Oracles

### Background

In the single-input case, a Boolean function takes one input \(x\) and returns either `True` or `False`.

There are four possible functions:

- \(f(x) = 0\) = always returns False (constant)  
- \(f(x) = 1\) = always returns True (constant)  
- \(f(x) = x\) = returns the input value (balanced)  
- \(f(x) = \neg x\) = returns the opposite of the input (balanced)  

In quantum computing, these functions are implemented as **oracles**.

The oracle must perform the transformation:

\[
U_f |x⟩|y⟩ = |x⟩ |y \oplus f(x)⟩
\]

This means the second qubit is flipped if and only if the function output is 1.

### Implementation

Each oracle is implemented using a simple quantum circuit with two qubits:

- qubit 0 = input \(|x⟩\)  
- qubit 1 = output \(|y⟩\)  

The logic is:

- For \(f(x) = 0\): do nothing (no gates needed)  
- For \(f(x) = 1\): apply an X gate to always flip the output  
- For \(f(x) = x\): use a CNOT gate so the output flips when \(x = 1\)  
- For \(f(x) = \neg x\): flip the output first, then apply CNOT  

These circuits directly implement the required transformation \(y \oplus f(x)\).

In [5]:
from qiskit import QuantumCircuit

def deutsch_oracle(name: str) -> QuantumCircuit:
    """
    Construct a 2-qubit oracle for Deutsch's algorithm.

    Qubit 0: input |x⟩
    Qubit 1: output |y⟩ (target qubit)
    """
    qc = QuantumCircuit(2, name=f"U_{name}")

    if name == "f0":
        # f(x) = 0  do nothing
        pass

    elif name == "f1":
        # f(x) = 1  always flip y
        qc.x(1)

    elif name == "fx":
        # f(x) = x  flip y if x = 1
        qc.cx(0, 1)

    elif name == "fnotx":
        # f(x) = ¬x effectively y ⊕ (1 ⊕ x)
        qc.x(1)
        qc.cx(0, 1)

    else:
        raise ValueError("Invalid oracle name: use 'f0', 'f1', 'fx', or 'fnotx'.")

    return qc


### Testing the Oracles

Each oracle is tested using different input values for \(x\) and \(y\).

This allows us to clearly see how the output qubit changes and confirms that the oracle is working correctly.

In [6]:
for name in ["f0", "f1", "fx", "fnotx"]:
    oracle = deutsch_oracle(name)
    print(f"\nOracle: {name}")
    print(oracle.draw())  # ASCII diagram



Oracle: f0
     
q_0: 
     
q_1: 
     

Oracle: f1
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

Oracle: fx
          
q_0: ──■──
     ┌─┴─┐
q_1: ┤ X ├
     └───┘

Oracle: fnotx
               
q_0: ───────■──
     ┌───┐┌─┴─┐
q_1: ┤ X ├┤ X ├
     └───┘└───┘


### Demonstration

To verify the oracles work correctly, each one is run on all four computational basis states: `|00⟩`, `|01⟩`, `|10⟩`, `|11⟩`.

For each input `|x⟩|y⟩`, the oracle should produce `|x⟩|y ⊕ f(x)⟩` - meaning the input qubit stays the same and the output qubit gets flipped only when `f(x) = 1`.

The table below shows what the output qubit should be for each oracle and each input combination.

In [7]:
from qiskit_aer import AerSimulator
from qiskit import transpile

sim_statevector = AerSimulator(method="statevector")

print(f"{'Oracle':<8} {'|x>|y>':<8} {'Result':<8}  (x stays same, y flips if f(x)=1)")
print("-" * 55)

for name in ["f0", "f1", "fx", "fnotx"]:
    oracle = deutsch_oracle(name)
    for x in [0, 1]:
        for y in [0, 1]:
            # Build a small test circuit: set |x>|y>, apply oracle, measure
            test = QuantumCircuit(2, 2)
            if x == 1:
                test.x(0)   # set input qubit to |1>
            if y == 1:
                test.x(1)   # set output qubit to |1>
            test.append(oracle.to_gate(), [0, 1])
            test.measure([0, 1], [0, 1])

            compiled = transpile(test, sim_statevector)
            counts = sim_statevector.run(compiled, shots=1).result().get_counts()
            result = list(counts.keys())[0]  # only one outcome possible

            # Qiskit bit order is reversed: result[0] = qubit 1, result[1] = qubit 0
            out_x = result[1]
            out_y = result[0]
            print(f"{name:<8} |{x}⟩|{y}⟩  ->  |{out_x}⟩|{out_y}⟩")
    print()

Oracle   |x>|y>   Result    (x stays same, y flips if f(x)=1)
-------------------------------------------------------
f0       |0⟩|0⟩  ->  |0⟩|0⟩
f0       |0⟩|1⟩  ->  |0⟩|1⟩
f0       |1⟩|0⟩  ->  |1⟩|0⟩
f0       |1⟩|1⟩  ->  |1⟩|1⟩

f1       |0⟩|0⟩  ->  |0⟩|1⟩
f1       |0⟩|1⟩  ->  |0⟩|0⟩
f1       |1⟩|0⟩  ->  |1⟩|1⟩
f1       |1⟩|1⟩  ->  |1⟩|0⟩

fx       |0⟩|0⟩  ->  |0⟩|0⟩
fx       |0⟩|1⟩  ->  |0⟩|1⟩
fx       |1⟩|0⟩  ->  |1⟩|1⟩
fx       |1⟩|1⟩  ->  |1⟩|0⟩

fnotx    |0⟩|0⟩  ->  |0⟩|1⟩
fnotx    |0⟩|1⟩  ->  |0⟩|0⟩
fnotx    |1⟩|0⟩  ->  |1⟩|0⟩
fnotx    |1⟩|1⟩  ->  |1⟩|1⟩



### Discussion

All four oracles produce the correct output for every input combination, so the transformation `|x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩` is working as expected.

The `f0` oracle is worth pointing out, it contains no gates at all, which might look like a mistake at first glance. But it is correct: if `f(x) = 0` for all inputs, then `y ⊕ 0 = y`, so the output qubit should never change. An empty circuit is the right answer here.

The other thing worth mentioning is that these oracles are kept simple on purpose. The phase kickback effect that the algorithm depends on comes from preparing the ancilla qubit in the `|−⟩` state before calling the oracle, that preparation happens in the main Deutsch circuit, not inside the oracle itself. So the oracles just need to implement the XOR transformation correctly, which all four of them do.

### References

- Deutsch, D. (1985). *Quantum Theory, the Church–Turing Principle and the Universal Quantum Computer*. Proceedings of the Royal Society of London A, 400, 97–117.  
  The paper that introduced the quantum oracle model, the transformation `|x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩` used in this problem comes directly from Deutsch's original formulation.

- Nielsen, M. A., & Chuang, I. L. (2010). *Quantum Computation and Quantum Information*. Cambridge University Press.  
  Chapter 1 explains the oracle model clearly and was useful for understanding why the four single-bit functions are the only possible cases.

- IBM Quantum, *Deutsch's Algorithm*:  
  https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm  
  Used to understand how oracles are structured in Qiskit and how the gate transformation maps to the circuit.

- Qiskit Documentation, *QuantumCircuit*:  
  https://qiskit.org/documentation/  
  Used when building the oracle circuits and working with X and CNOT gates.

## Problem 4: Deutsch's Algorithm with Qiskit


### Background

Deutsch’s algorithm is one of the simplest examples of a quantum algorithm.

It solves the problem of deciding whether a single input Boolean function is **constant** or **balanced**.

For a classical algorithm, this can require checking more than one input.
In the quantum case, the answer can be found using only **one oracle query**.

The key idea is that the quantum circuit uses:

- **superposition** to evaluate multiple possibilities at once
- **interference** to combine amplitudes in a way that reveals a global property of the function

### Implementation

The circuit uses two qubits:

- the first qubit stores the input
- the second qubit is an ancilla qubit used by the oracle

The steps are:

1. prepare the state \(|0⟩|1⟩\)
2. apply Hadamard gates to both qubits
3. apply the oracle once
4. apply a Hadamard gate to the input qubit
5. measure the input qubit

The final measurement tells us the function type:

- result `0` means the function is **constant**
- result `1` means the function is **balanced**

In [8]:
from qiskit_aer import AerSimulator

sim = AerSimulator()

def deutsch_algorithm(oracle):
    qc = QuantumCircuit(2, 1)

    # Step 1: Prepare |0⟩|1⟩
    qc.x(1)

    # Step 2: Hadamards
    qc.h(0)
    qc.h(1)

    # Step 3: Oracle
    qc.append(oracle.to_gate(), [0, 1])

    # Step 4: Hadamard on input qubit
    qc.h(0)

    # Step 5: Measure input qubit
    qc.measure(0, 0)

    return qc

### Demonstration

To test the circuit, I ran Deutsch’s algorithm using all four of the oracles from Problem 3:

- \(f(x) = 0\)
- \(f(x) = 1\)
- \(f(x) = x\)
- \(f(x) = \neg x\)

The measurement counts show whether the algorithm classifies each function as constant or balanced.

This allows the result of the quantum circuit to be compared directly with the known behaviour of each oracle.

In [9]:
from qiskit import transpile

def classify(counts):
    return "constant" if counts.get("0", 0) > counts.get("1", 0) else "balanced"

for name in ["f0", "f1", "fx", "fnotx"]:
    oracle = deutsch_oracle(name)
    circuit = deutsch_algorithm(oracle)

    # Compile custom oracle gate into backend-supported instructions.
    compiled = transpile(circuit, sim)
    result = sim.run(compiled, shots=1024).result()
    counts = result.get_counts()

    print(f"{name:7s} -> {counts} -> {classify(counts)}")

f0      -> {'0': 1024} -> constant
f1      -> {'0': 1024} -> constant
fx      -> {'1': 1024} -> balanced
fnotx   -> {'1': 1024} -> balanced


### Why the Algorithm Works

After the Hadamard gates, the input qubit is placed into a superposition of both possible input values.

The oracle then affects the phase of the state depending on the function.

When the final Hadamard gate is applied, the amplitudes interfere:

- for a **constant** function, the interference leads to measuring `0`
- for a **balanced** function, the interference leads to measuring `1`

So the algorithm does not find the value of the function for one specific input.
Instead, it determines a global property of the function.

The oracle imprints the function value into the phase of the superposition
(phase kickback).

After the final Hadamard gate, interference causes:

- Constructive interference at |0⟩ if the function is constant.
- Destructive interference at |0⟩ if the function is balanced.

Thus the measurement outcome deterministically reveals the function type
with only one oracle evaluation.

### Discussion

All four oracles are classified correctly, and every run produces 1024 out of 1024 shots on the same outcome. That is what you would expect from a noiseless simulator, the algorithm is deterministic, so there is no randomness in the result.

The comparison with the classical approach is what makes this interesting. For a single-bit function, classical worst case is 2 evaluations. Deutsch's algorithm always uses exactly 1, no matter which of the four functions it gets. That might not sound like a huge deal for just one bit, but the Deutsch–Jozsa algorithm in Problem 5 shows the same principle scaling to any number of input bits.

One thing I found useful to understand is why the ancilla qubit has to start in `|1⟩`. Applying Hadamard to `|1⟩` gives `|−⟩`, and it is this state that causes phase kickback when the oracle runs. If the ancilla started in `|0⟩` instead, the oracle would have nothing to kick back into the input qubit and the algorithm would not work.

### References

- Deutsch, D. (1985). *Quantum Theory, the Church–Turing Principle and the Universal Quantum Computer*. Proceedings of the Royal Society of London A, 400, 97–117.  
  The original paper introducing Deutsch's algorithm. This is the direct source for Problem 4, the circuit here is an implementation of exactly what Deutsch described.

- Nielsen, M. A., & Chuang, I. L. (2010). *Quantum Computation and Quantum Information*. Cambridge University Press.  
  Section 1.4.3 covers Deutsch's algorithm clearly, including why phase kickback works and how the interference produces a deterministic result with one oracle call.

- IBM Quantum, *Deutsch's Algorithm*:  
  https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/quantum-query-algorithms/deutsch-algorithm  
  Used to check the circuit structure and understand how the measurement result maps to constant vs balanced.

- Qiskit Documentation, *QuantumCircuit*:  
  https://qiskit.org/documentation/  
  Used when building the circuit and working out how to append the oracle as a gate using `to_gate()`.

## Problem 5: Scaling to the Deutsch–Jozsa Algorithm

### Background

The Deutsch–Jozsa algorithm is an extension of Deutsch’s algorithm.

Instead of working with a function that takes one input bit, it works with functions that take multiple input bits.

In this problem, the function takes **four Boolean inputs**, which means there are:

\[
2^4 = 16
\]

possible input combinations.

The function is guaranteed to be either:

- **constant** = always returns the same output  
- **balanced** = returns `True` for exactly half of the inputs  

The goal is to use a quantum circuit to determine the function type using only one oracle query.

### Example Functions

To test the algorithm, I created both constant and balanced functions.

Constant functions:

- always return `False`
- always return `True`

Balanced functions:

- returns the first input bit
- returns the parity of all four input bits

These functions allow the circuit to be tested on both possible outcomes.

In [10]:
# Constant functions - return the same value regardless of input
def const_false(*_):
    return False

def const_true(*_):
    return True

# Balanced functions
def balanced_first_bit(a, b, c, d):
    return a  # True for exactly the 8 inputs where a is True

def balanced_parity(a, b, c, d):
    return a ^ b ^ c ^ d  # True for exactly the 8 inputs with an odd number of True bits

### Oracle Construction

The classical function must be converted into a quantum oracle.

The oracle performs the transformation:

\[
U_f |x⟩|y⟩ = |x⟩ |y \oplus f(x)⟩
\]

To do this:

- all possible inputs are checked
- if the function returns `True`, the oracle flips the ancilla qubit
- multi-controlled X gates are used to match specific input combinations

This allows the classical function to be represented as a quantum circuit.

In [11]:
from itertools import product
from qiskit import QuantumCircuit

def build_uf(f, n=4):
    """
    Build a Deutsch–Jozsa oracle for a Boolean function f with n inputs.

    The oracle implements:
        |x>|y> -> |x>|y XOR f(x)|

    Qubits 0..n-1 are the input register.
    Qubit n is the ancilla/target qubit.
    """
    qc = QuantumCircuit(n + 1, name="U_f")

    inputs = list(product([False, True], repeat=n))

    for x in inputs:
        if f(*x):
            # Flip qubits where input bit is 0 so all controls become on-1 controls
            for i, bit in enumerate(x):
                if bit is False:
                    qc.x(i)

            # Multi-controlled X onto ancilla
            qc.mcx(list(range(n)), n)

            # Undo the flips
            for i, bit in enumerate(x):
                if bit is False:
                    qc.x(i)

    return qc

### Circuit Construction

The Deutsch–Jozsa circuit uses:

- 4 input qubits
- 1 ancilla qubit

The circuit works in the following steps:

1. prepare the ancilla in state \(|1⟩\)
2. apply Hadamard gates to create superposition
3. apply the oracle once
4. apply Hadamard gates again to the input qubits
5. measure the input register

The final measurement determines whether the function is constant or balanced.

In [12]:
def deutsch_jozsa_circuit(f, n=4):
    """
    Build the Deutsch–Jozsa circuit for an n-input Boolean function f.
    """
    oracle = build_uf(f, n)
    qc = QuantumCircuit(n + 1, n)

    # Prepare ancilla in |1>
    qc.x(n)

    # Apply Hadamard gates to all qubits
    for i in range(n + 1):
        qc.h(i)

    # Apply oracle
    qc.append(oracle.to_gate(), range(n + 1))

    # Apply Hadamard gates to input register only
    for i in range(n):
        qc.h(i)

    # Measure input register
    qc.measure(range(n), range(n))

    return qc

### Results

The circuit is tested using:

- two constant functions
- two balanced functions

The expected behaviour is:

- measuring `0000` = constant  
- measuring anything else = balanced  

The simulation results below show whether the circuit correctly identifies each function.

In [13]:
from qiskit_aer import AerSimulator

sim = AerSimulator()

def classify_deutsch_jozsa(counts):
    """
    If the measured result is all zeros, classify as constant.
    Otherwise classify as balanced.
    """
    most_common = max(counts, key=counts.get)

    if most_common == "0000":
        return "constant"
    else:
        return "balanced"

In [14]:
from qiskit import transpile

test_functions = [
    ("const_false", const_false, "constant"),
    ("const_true", const_true, "constant"),
    ("balanced_first_bit", balanced_first_bit, "balanced"),
    ("balanced_parity", balanced_parity, "balanced"),
]

for name, f, expected in test_functions:
    qc = deutsch_jozsa_circuit(f, n=4)
    # Compile custom/composite oracle instructions for Aer.
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=1024).result()
    counts = result.get_counts()
    predicted = classify_deutsch_jozsa(counts)

    print(f"{name:20s} expected={expected:8s} predicted={predicted:8s} counts={counts}")

const_false          expected=constant predicted=constant counts={'0000': 1024}
const_true           expected=constant predicted=constant counts={'0000': 1024}
balanced_first_bit   expected=balanced predicted=balanced counts={'0001': 1024}
balanced_parity      expected=balanced predicted=balanced counts={'1111': 1024}


### Discussion

The circuit correctly identifies all four functions, both constant functions produce `0000` and both balanced functions produce a non-zero result, which matches what the theory predicts.

The comparison with Problem 2 is the main point here. The classical algorithm needed up to 9 evaluations to be certain about a 4-bit function. The quantum algorithm always uses exactly 1 oracle call. For an *n*-bit function the classical worst case is 2^(n−1) + 1, while the quantum circuit stays at 1 regardless of how large *n* gets. That is a genuine exponential speedup, not just a minor improvement.

It is worth being honest about the limitations though. This speedup only applies because the function is guaranteed to be either constant or balanced, that is the promise the algorithm relies on. If the function were something arbitrary, the result would be meaningless. Deutsch–Jozsa is more of a proof of concept than a practical tool, but it was historically important because it was one of the first clear examples of a quantum algorithm solving a well-defined problem faster than any classical algorithm ever could.

### References

- Deutsch, D., & Jozsa, R. (1992). *Rapid Solution of Problems by Quantum Computation*. Proceedings of the Royal Society of London A, 439, 553–558.  
  The original Deutsch–Jozsa paper. It proves that a quantum algorithm can solve this problem with one oracle call where any classical deterministic algorithm needs exponentially many.

- Nielsen, M. A., & Chuang, I. L. (2010). *Quantum Computation and Quantum Information*. Cambridge University Press.  
  Section 1.4.3 walks through the Deutsch–Jozsa circuit and the proof of correctness. This was useful for checking that the circuit structure here matches the standard description.

- IBM Quantum, *Deutsch–Jozsa Algorithm*:  
  https://quantum.cloud.ibm.com/learning/en/modules/computer-science/deutsch-jozsa  
  Helpful for understanding how the algorithm scales to multiple input bits and how the oracle is structured in Qiskit specifically.

- Qiskit Documentation:  
  https://qiskit.org/documentation/  
  Used when working out how to build the multi-controlled X gate oracle and how to append a sub-circuit as a gate using `to_gate()`.